# Day 2.3 — Embeddings and Semantic Search

An embedding is a numerical representation used to compare approximate meaning. We embed chunks once, embed each question, then rank by similarity.

The embedding model runs locally; OpenRouter is used later for answer generation.

## Before you begin

### Learning outcomes

Compare deterministic teaching embeddings with sentence-transformer semantic search.

Architecture reference: [D06](../../diagrams/source/day_02.md).

### Expected observation

The offline hasher is stable but limited; the optional real embedder better handles paraphrases after download.


## Concept briefing

## What embeddings do - and do not do

An embedding converts text into a vector so that a similarity function can rank nearby
representations. A trained semantic embedding may place paraphrases close together. The
course's deterministic token-hash embedder is different: it maps token features into a
stable numeric space for offline orchestration tests. It cannot genuinely understand
meaning and must not be presented as a production semantic model.

Similarity answers "which candidates are closest under this representation?" It does
not prove that a passage is relevant, sufficient or correct. Scores from different
models are not directly comparable, and there is no universal threshold.


In [ ]:
# Run before Day 2 if needed:
# %pip install -q sentence-transformers
import os,sys
from pathlib import Path
here=Path.cwd().resolve(); candidates=[here,here/"day_02_knowledge_and_state",here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src"))
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import SentenceTransformerEmbedder
from knowledge_agent.retrieval import VectorIndex
chunks=load_markdown_corpus(project_root/"data"/"corpus")

In [ ]:
model_name=os.getenv("EMBEDDING_MODEL","sentence-transformers/all-MiniLM-L6-v2")
embedder=SentenceTransformerEmbedder(model_name)
index=VectorIndex(embedder)
index.add(chunks)
print(len(index.chunks),"chunks indexed")

In [ ]:
question="Which equipment remains energized away from the utility grid?"
for item in index.search(question,top_k=3):
    print(round(item.score,3),item.chunk.source,item.chunk.section)
    print(item.chunk.text[:140])

## What the score means

Similarity ranks candidates; it does not prove relevance or correctness. There is no universal score threshold. We evaluate retrieval on known questions rather than trusting an attractive decimal.

## Optional: place vectors in Chroma

Our in-memory index makes the mathematics visible. Chroma provides database storage and search interfaces around the same embeddings.

In [ ]:
# %pip install -q chromadb
from knowledge_agent.retrieval import ChromaVectorIndex
chroma_index=ChromaVectorIndex(embedder,collection_name="day2_lab")
chroma_index.add(chunks)
[(x.chunk.section,round(x.score,3)) for x in chroma_index.search(question,3)]

## Exercise and checkpoint

Compare keyword, in-memory semantic, and Chroma results for three questions. Explain which component creates vectors and which stores/searches them. Next, retrieved text becomes model context.

## Your turn

Record one query where both agree and one where they differ.

## Recap

Similarity is a ranking signal, not proof that a chunk answers the question.
